# notebook_01: Veri Keşfi ve Tanımlayıcı İstatistik

Bu çalışma kapsamında enerji perakende sektörüne ait elektrik tüketim ve tahsilat verilerinin genel yapısı incelenmiş, veri kalitesi kontrol edilmiş ve temel tanımlayıcı istatistikler hesaplanmıştır.

### Yapılan İşlemler:
1. Excel sayfalarındaki verilerin `.info()`, `.describe()` ve `.head()` metotları ile incelenmesi
2. İlçe bazında benzersiz müşteri sayılarının tespiti
3. Tüketim verilerinin birleştirilmesi
4. Eksik, negatif ve aykırı (outlier) değerlerin analizi
5. Hesap sınıflarına göre tüketim kırılımları

In [4]:
import pandas as pd
import numpy as np

file_path = r'../data/elektrik_veri_hashed.xlsx'
xls = pd.ExcelFile(file_path)

print("Sayfa isimleri:", xls.sheet_names)

print("Tahsilat sayfaları yükleniyor...")
df_tahsilat = pd.read_excel(xls, sheet_name='Tahsilat')
df_tahsilat_1 = pd.read_excel(xls, sheet_name='Tahsilat 1')

print("Tahakkuk sayfaları yükleniyor...")
df_tahakkuk = pd.read_excel(xls, sheet_name='Tahakkuk')        # Hamamözü
df_tahakkuk_1 = pd.read_excel(xls, sheet_name='Tahakkuk 1')    # Gümüşhacıköy
df_tahakkuk_2 = pd.read_excel(xls, sheet_name='Tahakkuk 2')    # Göynücek

print("Yükleme tamamlandı! Boyutlar kontrol ediliyor...")

print(f"Tahsilat: {df_tahsilat.shape}")
print(f"Tahsilat 1: {df_tahsilat_1.shape}")
print(f"Tahakkuk: {df_tahakkuk.shape}")
print(f"Tahakkuk 1: {df_tahakkuk_1.shape}")
print(f"Tahakkuk 2: {df_tahakkuk_2.shape}")

Sayfa isimleri: ['Tahsilat', 'Tahsilat 1', 'Tahakkuk', 'Tahakkuk 1', 'Tahakkuk 2']
Tahsilat sayfaları yükleniyor...
Tahakkuk sayfaları yükleniyor...
Yükleme tamamlandı! Boyutlar kontrol ediliyor...
Tahsilat: (636993, 9)
Tahsilat 1: (917632, 22)
Tahakkuk: (124818, 10)
Tahakkuk 1: (765657, 10)
Tahakkuk 2: (295223, 10)


In [5]:
dfs = {
    "Tahsilat": df_tahsilat,
    "Tahsilat 1": df_tahsilat_1,
    "Tahakkuk (Hamamözü)": df_tahakkuk,
    "Tahakkuk 1 (Gümüşhacıköy)": df_tahakkuk_1,
    "Tahakkuk 2 (Göynücek)": df_tahakkuk_2
}

for name, df in dfs.items():
    print(f"\n{'='*20} {name} Özet Bilgileri {'='*20}")
    df.info()
    print("\nTemel İstatistikler:")
    display(df.describe())
    print("\nİlk 3 Satır:")
    display(df.head(3))


==================== Tahsilat Özet Bilgileri ====================
<class 'pandas.DataFrame'>
RangeIndex: 636993 entries, 0 to 636992
Data columns (total 9 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   Şube                   636993 non-null  str           
 1   Kasa                   636993 non-null  str           
 2   İlçe                   636993 non-null  str           
 3   Söz.hsp.(bağımsız)     636993 non-null  int64         
 4   Tahsilat Tarihi        636993 non-null  datetime64[us]
 5   Nakit Tahsilat         523 non-null     float64       
 6   Mahsuben Tahsilat      7542 non-null    float64       
 7   Kredi Kartı Tahsilatı  0 non-null       float64       
 8   Banka Tahsilatı        628933 non-null  float64       
dtypes: datetime64[us](1), float64(4), int64(1), str(3)
memory usage: 43.7 MB

Temel İstatistikler:


,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
count,6.369930e+05,636993,523.000000,7542.000000,0.0,628933.000000
mean,5.019884e+09,2024-03-05 09:26:07.644856,694.966635,6180.182282,NaN,372.629109
min,1.758320e+05,2023-01-01 00:00:00,7.450000,-34508.950000,NaN,0.010000
25%,2.532887e+09,2023-07-28 00:00:00,425.330000,44.477500,NaN,120.000000
50%,5.008502e+09,2024-02-26 00:00:00,524.670000,290.410000,NaN,208.000000
75%,7.525722e+09,2024-09-30 00:00:00,688.830000,2729.110000,NaN,322.000000
max,9.999600e+09,2025-05-31 00:00:00,11373.740000,399526.780000,NaN,606473.800000
std,2.884435e+09,NaN,758.319428,23828.022593,NaN,3265.430202



İlk 3 Satır:


,Şube,Kasa,İlçe,Söz.hsp.(bağımsız),Tahsilat Tarihi,Nakit Tahsilat,Mahsuben Tahsilat,Kredi Kartı Tahsilatı,Banka Tahsilatı
0,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2023-11-06,NaN,8648.95,NaN,NaN
1,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-06-26,NaN,762.40,NaN,NaN
2,Tayin edilmedi,Tayin edilmedi,TAŞOVA,4989745446,2024-07-10,NaN,311.60,NaN,NaN



==================== Tahsilat 1 Özet Bilgileri ====================
<class 'pandas.DataFrame'>
RangeIndex: 917632 entries, 0 to 917631
Data columns (total 22 columns):
 #   Column                                Non-Null Count   Dtype  
---  ------                                --------------   -----  
 0   Mali yıl/dönem                        917632 non-null  str    
 1   İl                                    917632 non-null  str    
 2   İlçe                                  917632 non-null  str    
 3   Söz.hsp.(bağımsız)                    917632 non-null  int64  
 4   Hesap Sınıfı                          917632 non-null  str    
 5   Tahakkuk Tutar                        917632 non-null  float64
 6   Son Ödeme Tarihinden Önceki Tahsilat  623908 non-null  float64
 7   Son Ödeme Tarihindeki Tahsilat        328193 non-null  float64
 8   Son Ödeme (1)                         20902 non-null   float64
 9   Son Ödeme (2)                         21664 non-null   float64
 10  Son Ödeme 

...

,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-01-01,2023-01-11,2023-03-06,2023-01-23,M001,Mesken,21.85
1,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-01-01,2023-02-10,2023-05-11,2023-02-20,M001,Mesken,44.50
2,AMASYA,GÜMÜŞHACIKÖY,7444449517,2023-02-01,2023-02-10,2023-05-11,2023-02-20,M001,Mesken,22.25



==================== Tahakkuk 2 (Göynücek) Özet Bilgileri ====================
<class 'pandas.DataFrame'>
RangeIndex: 295223 entries, 0 to 295222
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   il                 295223 non-null  str    
 1   ilce               295223 non-null  str    
 2   sozlesme_hesap_no  295223 non-null  int64  
 3   mali_yil_donem     295223 non-null  str    
 4   fatura_tarihi      295223 non-null  str    
 5   kayit_tarihi       295223 non-null  str    
 6   vade_tarihi        295223 non-null  str    
 7   hesap_sinifi       295223 non-null  object 
 8   Hesap Sınıfı       295223 non-null  str    
 9   kwh                295223 non-null  float64
dtypes: float64(1), int64(1), object(1), str(7)
memory usage: 22.5+ MB

Temel İstatistikler:


,sozlesme_hesap_no,kwh
count,2.952230e+05,295223.000000
mean,4.950566e+09,89.669891
std,2.887724e+09,742.276369
min,1.118550e+06,-4208.640000
25%,2.432854e+09,17.860000
50%,4.928206e+09,45.090000
75%,7.475616e+09,77.140000
max,9.997185e+09,105687.690000



İlk 3 Satır:


,il,ilce,sozlesme_hesap_no,mali_yil_donem,fatura_tarihi,kayit_tarihi,vade_tarihi,hesap_sinifi,Hesap Sınıfı,kwh
0,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2023-01-14,2023-03-06,2023-01-24,M001,Mesken,0.10
1,AMASYA,GÖYNÜCEK,9374624783,2023-01-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.12
2,AMASYA,GÖYNÜCEK,9374624783,2023-02-01,2025-03-12,2025-05-09,2025-03-24,M001,Mesken,0.20


In [6]:
print(f"Hamamözü Benzersiz Müşteri Sayısı: {df_tahakkuk['sozlesme_hesap_no'].nunique()}")
print(f"Gümüşhacıköy Benzersiz Müşteri Sayısı: {df_tahakkuk_1['sozlesme_hesap_no'].nunique()}")
print(f"Göynücek Benzersiz Müşteri Sayısı: {df_tahakkuk_2['sozlesme_hesap_no'].nunique()}")

# Tüm tüketim verilerini birleştirme
df_tahakkuk_all = pd.concat([df_tahakkuk, df_tahakkuk_1, df_tahakkuk_2], ignore_index=True)
print(f"\nBirleştirilmiş Tahakkuk Toplam Kayıt Sayısı: {df_tahakkuk_all.shape[0]}")

Hamamözü Benzersiz Müşteri Sayısı: 2981
Gümüşhacıköy Benzersiz Müşteri Sayısı: 18190
Göynücek Benzersiz Müşteri Sayısı: 7128

Birleştirilmiş Tahakkuk Toplam Kayıt Sayısı: 1185698


In [7]:
missing_kwh = df_tahakkuk_all['kwh'].isna().sum()
negative_kwh = (df_tahakkuk_all['kwh'] < 0).sum()

print(f"Eksik kwh Değeri Sayısı: {missing_kwh}")
print(f"Negatif kwh Değeri Sayısı: {negative_kwh}")

# IQR ile Outlier Hesabı
Q1 = df_tahakkuk_all['kwh'].quantile(0.25)
Q3 = df_tahakkuk_all['kwh'].quantile(0.75)
IQR = Q3 - Q1
alt_sinir = Q1 - 1.5 * IQR
ust_sinir = Q3 + 1.5 * IQR

outliers = df_tahakkuk_all[(df_tahakkuk_all['kwh'] < alt_sinir) | (df_tahakkuk_all['kwh'] > ust_sinir)]
print(f"Aykırı (Outlier) Kayıt Sayısı: {outliers.shape[0]}")
print(f"Belirlenen Üst Sınır: {ust_sinir:.2f} kWh")

hesap_sinifi_analizi = df_tahakkuk_all.groupby('hesap_sinifi')['kwh'].agg(['mean', 'median', 'std', 'count']).reset_index()
hesap_sinifi_analizi.columns = ['Hesap Sınıfı', 'Ortalama kWh', 'Medyan kWh', 'Standart Sapma', 'Kayıt Sayısı']

display(hesap_sinifi_analizi.sort_values(by='Kayıt Sayısı', ascending=False))

Eksik kwh Değeri Sayısı: 0
Negatif kwh Değeri Sayısı: 151
Aykırı (Outlier) Kayıt Sayısı: 48554
Belirlenen Üst Sınır: 172.98 kWh


,Hesap Sınıfı,Ortalama kWh,Medyan kWh,Standart Sapma,Kayıt Sayısı
8,M001,55.590010,47.360,73.902931,1026609
18,T001,167.827953,41.600,781.762054,91298
33,TA01,541.608449,16.150,3627.959619,17266
9,M002,81.283512,56.140,111.164648,10173
30,T019,688.441598,23.860,3911.912974,8500
19,T002,136.952881,28.420,798.495578,6706
32,T021,29.965663,15.760,41.331395,3505
5,A001,32.435252,13.940,112.225490,3189
17,SE01,87.035189,76.540,56.599681,2671
28,T013,676.935305,208.655,1212.334580,2460
